In [ ]:
# to print nvidia driver's status table + to confirm that a gpu is attached to this session
!nvidia-smi

In [ ]:
from google.colab import userdata, drive
import os

USERNAME = "mardyweb"
REPO     = "atml-pa0"
TOKEN    = userdata.get('GITHUB_TOKEN')

# storing the url in an environment variable:
os.environ['GIT_URL'] = f"https://{TOKEN}@github.com/{USERNAME}/{REPO}.git"

# clones the repo only if it is not already here (safe to re-run):
if not os.path.exists(f"/content/{REPO}"):
    !git clone $GIT_URL
%cd /content/$REPO

!git config user.email "maryamw17@outlook.com"
!git config user.name "Maryam"

In [ ]:
from utils import set_seed, get_device, subset_loaders, save_results, save_fig
import matplotlib.pyplot as plt

# should print cuda:
print(get_device())

In [ ]:
#task 2.1
!pip install -q transformers

from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch, requests
from utils import set_seed, get_device, save_results, save_fig

set_seed(42)
device = get_device()

# ViT-Base/16: 86M params, 12 encoder layers, 12 attention heads per layer, 768-dim embeddings, 16x16 patches, pre-trained on ImageNet-21k and
# fine-tuned on ImageNet-1k (so it predicts the standard 1000 classes)
MODEL = "google/vit-base-patch16-224"

# The processor handles resizing to 224x224 and normalising with the exact mean/std the model was trained on:
processor = ViTImageProcessor.from_pretrained(MODEL)
model_vit = ViTForImageClassification.from_pretrained(
    MODEL,
    attn_implementation="eager"     # SDPA is faster but hides attention weights
).to(device)

model_vit.eval()

print(f"patch size: {model_vit.config.patch_size}")
print(f"hidden dim: {model_vit.config.hidden_size}")
print(f"layers: {model_vit.config.num_hidden_layers}, "
      f"heads: {model_vit.config.num_attention_heads}")
print(f"output classes: {model_vit.config.num_labels}")

In [ ]:
from google.colab import files
uploaded = files.upload()   # opens a file picker so i can select 3 images

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

images = {}
for fname in uploaded:
    # "mycat.jpg" -> "mycat" (removes file type)
    name = fname.rsplit('.', 1)[0]
    images[name] = Image.open(fname).convert("RGB")
    print(f"{name}: {images[name].size}")

fig, ax = plt.subplots(1, len(images), figsize=(4*len(images), 4))
if len(images) == 1: ax = [ax]
for a, (name, im) in zip(ax, images.items()):
    a.imshow(im); a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
results_cls = {}

for name, img in images.items():
    # return_tensors="pt" gives PyTorch tensors, the processor resizes to 224x224 and normalises
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_vit(**inputs)

    # logits: [1, 1000]: one raw score per ImageNet class.
    logits = outputs.logits
    probs  = logits.softmax(dim=-1)[0]

    top5 = probs.topk(5)
    top1_idx = top5.indices[0].item()
    # id2label maps class index to a human-readable name
    top1_label = model_vit.config.id2label[top1_idx]

    print(f"\n{name}  ->  {top1_label}  ({top5.values[0].item():.4f})")
    print("  top-5:")
    for p, i in zip(top5.values, top5.indices):
        print(f"    {model_vit.config.id2label[i.item()]:<35} {p.item():.4f}")

    results_cls[name] = {
        "top1_label": top1_label,
        "top1_prob": round(top5.values[0].item(), 4),
        "top5": [{"label": model_vit.config.id2label[i.item()],
                  "prob": round(p.item(), 4)}
                 for p, i in zip(top5.values, top5.indices)]
    }

save_results("task2_classification", results_cls)

In [ ]:
#task 2.2
import numpy as np
import torch.nn.functional as F

def get_cls_attention(img, layer=-1):
    """
    Returns the [CLS] token's attention over the 196 patches, as a 14x14 map.

    Walks through these steps:
      run the model with output_attentions=True -> take the final layer's attention matrix -> average over the 12 heads -> pull out the CLS row
      -> reshape to 14x14
    """
    inputs = processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        # output_attentions=True makes the model return every layer's attention weights alongside the logits.
        out = model_vit(**inputs, output_attentions=True)

    # out.attentions is a tuple of 12 tensors (one per encoder layer).
    # Each is [batch, heads, seq_len, seq_len] = [1, 12, 197, 197].
    # seq_len is 197 = 196 patches + 1 CLS token.
    att = out.attentions[layer]                 # final layer by default

    # Average over the 12 heads -> [1, 197, 197]. Each head attends to
    # different things so averaging gives one summary map.
    att = att.mean(dim=1)

    # Row 0 is the CLS token's attention to every token. Columns 1: drops CLS's attention to itself, leaving its attention to the 196 patches.
    cls_att = att[0, 0, 1:]                     # -> [196]

    # The 196 patches came from a 14x14 grid scanned row by row, so reshaping restores their spatial arrangement.
    return cls_att.reshape(14, 14).cpu().numpy(), out.logits


att_maps = {}
for name, img in images.items():
    att_maps[name], _ = get_cls_attention(img)
    a = att_maps[name]
    print(f"{name}: mean={a.mean():.4f} max={a.max():.4f} "
          f"min={a.min():.4f} std={a.std():.4f}")

In [ ]:
def overlay_attention(img, att_map, alpha=0.5):
    """
    Upsamples the 14x14 map to 224x224 and returns (resized image, upsampled map).
    Bicubic interpolation smooths the blocky patch grid into a continuous heatmap.
    """
    img_resized = img.resize((224, 224))

    # interpolate expects [batch, channels, H, W], so add two dummy dims
    t = torch.tensor(att_map)[None, None]
    up = F.interpolate(t, size=(224, 224), mode="bicubic",
                       align_corners=False)[0, 0].numpy()

    # normalise to [0,1] so the colour scale uses the full range
    up = (up - up.min()) / (up.max() - up.min() + 1e-8)
    return img_resized, up


fig, axes = plt.subplots(len(images), 3, figsize=(12, 4*len(images)))
if len(images) == 1: axes = axes[None, :]

for row, (name, img) in enumerate(images.items()):
    img_r, up = overlay_attention(img, att_maps[name])

    axes[row, 0].imshow(img_r)
    axes[row, 0].set_title(f"{name} — original"); axes[row, 0].axis("off")

    # raw attention map, no image, and shows the structure clearly
    im = axes[row, 1].imshow(att_maps[name], cmap="hot")
    axes[row, 1].set_title("attention map (14x14)"); axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)

    # brighter red means more attention
    axes[row, 2].imshow(img_r)
    axes[row, 2].imshow(up, cmap="jet", alpha=alpha if (alpha:=0.5) else 0.5)
    axes[row, 2].set_title("attention overlay"); axes[row, 2].axis("off")

plt.tight_layout()
save_fig(fig, "task2_attention_overlay")
plt.show()

# high std would mean the model is discriminating strongly between informative and uninformative patches
save_results("task2_attention_stats", {
    name: {"mean": float(a.mean()), "max": float(a.max()),
           "min": float(a.min()), "std": float(a.std())}
    for name, a in att_maps.items()
})

In [ ]:
# Does attention become less object-focused with depth? By layer 12 the patch
# representations have mixed information globally through 12 rounds of
# self-attention, so "where CLS looks" may no longer map onto "where the object is".
IMG_KEY = "image2 (1)"      # <-- replace with a real key from the print above

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for i, L in enumerate([2, 5, 8, 11]):
    a, _ = get_cls_attention(images[IMG_KEY], layer=L)
    ax[i].imshow(a, cmap="hot")
    ax[i].set_title(f"layer {L+1}")     # +1 because layers are 0-indexed
    ax[i].axis("off")

plt.tight_layout()
save_fig(fig, "task2_attention_by_layer")
plt.show()

def rollout(img):
    """
    Attention rollout (Abnar & Zuidema 2020): multiply the attention matrices
    across all 12 layers to trace how information actually reaches the CLS
    token, rather than only reading the final layer.

    Adding the identity accounts for residual connections: each token keeps
    a path to its own previous value, which raw attention weights ignore.
    """
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        atts = model_vit(**inputs, output_attentions=True).attentions

    eye = torch.eye(197).to(device)
    result = eye
    for a in atts:
        a = a.mean(1)[0]                    # average the 12 heads -> [197,197]
        a = a + eye                         # residual connection
        a = a / a.sum(-1, keepdim=True)     # renormalise so rows sum to 1
        result = a @ result                 # compose layer by layer

    return result[0, 1:].reshape(14, 14).cpu().numpy()   # CLS row, drop self


fig, ax = plt.subplots(2, len(images), figsize=(5*len(images), 9))
for col, (name, img) in enumerate(images.items()):
    ax[0, col].imshow(att_maps[name], cmap="hot")
    ax[0, col].set_title(f"{name}\nfinal layer only", fontsize=10)
    ax[0, col].axis("off")

    ax[1, col].imshow(rollout(img), cmap="hot")
    ax[1, col].set_title("rollout (all layers)", fontsize=10)
    ax[1, col].axis("off")

plt.tight_layout()
save_fig(fig, "task2_attention_rollout")
plt.show()

In [ ]:
#task 2.3
def get_head_attentions(img, layer=-1):
    """
    Same as get_cls_attention but without averaging over heads.
    Returns [12, 14, 14], one attention map per head.

    averaging hides specialisation
    """
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model_vit(**inputs, output_attentions=True)

    att = out.attentions[layer][0]      # [12 heads, 197, 197]
    cls_att = att[:, 0, 1:]             # each head's CLS row, patches only -> [12, 196]
    return cls_att.reshape(12, 14, 14).cpu().numpy()


IMG_KEY = "image1 (1)"      # the guitar as rollout localised it best
heads = get_head_attentions(images[IMG_KEY])

fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for h in range(12):
    ax = axes[h // 4, h % 4]
    ax.imshow(heads[h], cmap="hot")
    # std measures how peaked a head is: high std = attends sharply to a few patches, low std = spreads attention evenly (uninformative).
    ax.set_title(f"Head {h}  (std={heads[h].std():.4f})", fontsize=9)
    ax.axis("off")

plt.suptitle(f"Final-layer CLS attention per head — {IMG_KEY}", fontsize=12)
plt.tight_layout()
save_fig(fig, "task2_attention_heads")
plt.show()

In [ ]:
# to determine if specialisation is occurring:
#
#  1. Entropy (how spread out is a head's attention?) Low entropy = focused on
#     few patches; high entropy = diffuse and uninformative.
#  2. Pairwise correlation: if all heads produced the same map, they'd be
#     redundant. Low correlation between heads = they attend to different things = specialisation.

def entropy(a):
    p = a.flatten() / a.sum()
    return float(-(p * np.log(p + 1e-12)).sum())

ent = [entropy(heads[h]) for h in range(12)]
max_ent = np.log(196)     # uniform attention over all 196 patches

print(f"max possible entropy (uniform): {max_ent:.3f}\n")
print(f"{'head':<6} {'entropy':>9} {'std':>9} {'peak patch':>12}")
for h in range(12):
    peak = np.unravel_index(heads[h].argmax(), (14, 14))
    print(f"{h:<6} {ent[h]:>9.3f} {heads[h].std():>9.4f} {str(peak):>12}")

# Correlation between every pair of heads
flat = heads.reshape(12, -1)
corr = np.corrcoef(flat)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(12)); ax.set_yticks(range(12))
ax.set_xlabel("head"); ax.set_ylabel("head")
ax.set_title("Pairwise correlation between head attention maps")
plt.colorbar(im)
plt.tight_layout()
save_fig(fig, "task2_head_correlation")
plt.show()

off_diag = corr[~np.eye(12, dtype=bool)]
print(f"\nmean off-diagonal correlation: {off_diag.mean():.3f}")
print(f"most similar pair:   {np.unravel_index(np.argmax(corr - np.eye(12)*2), (12,12))}")
print(f"most dissimilar pair: {np.unravel_index(np.argmin(corr), (12,12))}")

save_results("task2_head_specialisation", {
    "image": IMG_KEY,
    "entropy": [round(e, 4) for e in ent],
    "std": [round(float(heads[h].std()), 5) for h in range(12)],
    "max_entropy_uniform": round(float(max_ent), 4),
    "mean_offdiag_correlation": round(float(off_diag.mean()), 4),
})

In [ ]:
# committing + pushing to github
!git config --global core.editor true
!git add .
!git commit -m "Task 2.3"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL

In [ ]:
from google.colab import _message
import json, os

nb = _message.blocking_request('get_ipynb', timeout_sec=60)['ipynb']
os.makedirs('notebooks', exist_ok=True)
with open('notebooks/task2_vit.ipynb', 'w') as f:
    json.dump(nb, f, indent=1)

!git add notebooks/task2_vit.ipynb
!git commit -m "implemented 2.3"
!git pull --no-rebase --no-edit $GIT_URL main
!git push $GIT_URL